# Industrial Conveyor Reliability: forecasting tool (Req. 2 + Req. 3)

**Team:** TEAM MEMBER NAMES

**How to run:** set `INPUT_PATH` to a Parquet file with one conveyor's daily history (same columns as the fleet file), or leave it empty to upload one, then *Runtime → Run all*. The tool uses every row, sorts by `Date`, detects the last observed day itself, and forecasts:
- **Req. 2:** the next five failures: component, expected date, days from the last observed day, P10–P90 dates;
- **Req. 3:** expected downtime per month for the 3 years after the last observed day, the cumulative curve, and the expected total with an 80% interval.

**Method.** Supervised models on features computed *as of* the last observed day (component ages on their own clocks, per-bearing-position ages, own failure rates, condition-monitoring trends, load class, seasonal climate). Training origins are every week of every fleet conveyor from day 200 on, labelled with what actually happened next: days to each of the next 5 failures (XGBoost AFT, censoring-aware), the component of each (LightGBM multiclass), and failures per future month (LightGBM Poisson). Hyperparameters were tuned with Optuna on grouped cross-validation (appendix); 5 seeds are bagged. The P10/P90 bands are empirical quantiles of out-of-fold errors per load class.

**Expert rule on top of the ML (fleet observation):** speed sensors have a hard design-life ceiling of ≈50,000 operating hours (lives end at 49,980–50,016 h), so when the input conveyor's sensor will reach it before the 5th predicted failure, a Speed_Sensor failure is inserted at that date (`Source = Rule`).

**Downtime** = 36 h per failure (12 h failure day + one 24 h corrective day, constant in the data) + 24 h per planned-maintenance day (3 per year: 15 Mar, 15 Jul, 15 Dec). Corrective and planned are reported separately and as a total.

**Models.** By default the notebook loads the models fitted by `train_all` (section 3) from a pinned commit and checks every file's SHA-256, so the result cannot change after submission. Set `RETRAIN_FROM_FLEET = True` to rebuild them here from the fleet file with the same code instead.

## 1. Input

In [ ]:
INPUT_PATH = "https://raw.githubusercontent.com/Zoltan000/ReliabilityAndSafetyCompetition/d5021b7b22f36607c0fe2f5497ad681144ab44cb/Example_P02CV27_6Years.parquet"  #@param {type:"string"}
# A local path, a mounted Drive path (/content/drive/...), or an http(s) URL. Leave empty to upload a file.

RETRAIN_FROM_FLEET = False  #@param {type:"boolean"}
FLEET_PATH = ""  #@param {type:"string"}
# Only used when RETRAIN_FROM_FLEET is True: path or URL (Google Drive links work) of
# 2026-09_compet_Student_Historical_Data_V03.parquet. Retraining fits 10 models (5 CV folds for the
# calibration + 5 final seeds): ~35 min on a 22-core PC, several hours on a free 2-core Colab runtime.

ARTIFACTS_BASE = "https://raw.githubusercontent.com/Zoltan000/ReliabilityAndSafetyCompetition/d5021b7b22f36607c0fe2f5497ad681144ab44cb"  # pinned commit: fitted models cannot change after submission

## 2. Environment

In [ ]:
!pip -q install lightgbm==4.7.0 xgboost==3.2.0 pyarrow
import os, sys, json, hashlib, urllib.request
os.makedirs('conveyor', exist_ok=True)

## 3. Workflow source code
Identical to the tested package `src/conveyor/` in the submission repository:
`io` (load + validate one conveyor), `features` (as-of features, causal), `labels` (future targets, training only), `evaluate` (sealed split, grouped CV, metrics), `models` (baselines + the direct ML model), `train` (fleet file → tables → calibration → final models), `forecast` (the tool).

In [ ]:
%%writefile conveyor/__init__.py
"""Conveyor reliability forecasting package."""


In [ ]:
%%writefile conveyor/io.py
"""Load and validate one conveyor's (or the fleet's) daily history."""
from __future__ import annotations

import numpy as np
import pandas as pd

COMPONENTS = ["Bearing", "Conveyor_Belt", "Motor_Reducer", "Speed_Sensor",
              "Controller_PC", "Control_Software", "Contactor"]
STATES = ["RUNNING", "FAILURE_DAY", "CORRECTIVE_DOWNTIME", "PLANNED_MAINTENANCE"]
REQUIRED = ["Date", "Conveyor_ID", "Plant_ID", "Daily_State", "Failure_Type", "Failed_Component_ID",
            "Age_Calendar_Days", "Bearing_Count", "Cumulative_Operating_Hours", "Cumulative_kg",
            "Cumulative_Start_Stop_Cycles", "Downtime_Hours_Day"]
STATIC = ["Plant_ID", "Conveyor_ID", "EIS_Date", "Length_m", "Bearing_Count", "Load_Class",
          "Rated_Throughput_kg_h", "Belt_Speed_m_s", "Roller_Diameter_mm"]


def clean(df: pd.DataFrame) -> pd.DataFrame:
    """Validate one conveyor's rows: parse dates, sort, dedupe, and fill missing calendar days.

    Uses every row supplied; the last observed day is simply the max Date.
    """
    missing = [c for c in REQUIRED if c not in df.columns]
    if missing:
        raise ValueError(f"Input is missing required columns: {missing}")
    df = df.copy()
    df["Date"] = pd.to_datetime(df["Date"]).dt.normalize()
    if df["Conveyor_ID"].nunique() != 1:
        raise ValueError("Input must contain exactly one conveyor")
    df = df.sort_values("Date").drop_duplicates("Date", keep="last").reset_index(drop=True)
    full = pd.date_range(df["Date"].iloc[0], df["Date"].iloc[-1], freq="D")
    if len(full) != len(df):  # calendar gaps: reindex, carry static/cumulative fields forward
        df = df.set_index("Date").reindex(full).rename_axis("Date").reset_index()
        ffill_cols = STATIC + ["Cumulative_Operating_Hours", "Cumulative_kg",
                               "Cumulative_Start_Stop_Cycles", "Cumulative_Failure_Count"]
        df[ffill_cols] = df[ffill_cols].ffill()
        df["Daily_State"] = df["Daily_State"].fillna("RUNNING")
        df["Age_Calendar_Days"] = (df["Date"] - pd.to_datetime(df["EIS_Date"])).dt.days
    return df


def load_conveyor(path: str) -> pd.DataFrame:
    return clean(pd.read_parquet(path))


def load_fleet(path: str) -> pd.DataFrame:
    df = pd.read_parquet(path)
    df["Date"] = pd.to_datetime(df["Date"]).dt.normalize()
    return df.sort_values(["Conveyor_ID", "Date"]).reset_index(drop=True)


def iter_conveyors(fleet: pd.DataFrame):
    """Yield (conveyor_id, cleaned frame) for each conveyor in a fleet frame."""
    for cid, g in fleet.groupby("Conveyor_ID", sort=True):
        yield cid, clean(g)


def season_of(dates) -> np.ndarray:
    """Meteorological season index: 0=DJF, 1=MAM, 2=JJA, 3=SON (steps fall on the 1st of Mar/Jun/Sep/Dec)."""
    m = pd.DatetimeIndex(dates).month.to_numpy()
    return ((m % 12) // 3).astype(np.int8)


In [ ]:
%%writefile conveyor/features.py
"""As-of-origin features, identical for training (every origin day) and inference (last observed day).

Every feature on day i uses only rows 0..i of the conveyor's own history, plus the fixed
fleet climatology profile (4 seasonal temperature levels). Future weather is replaced by its
known expectation: the conveyor's own per-season temperature levels applied to the future calendar.
"""
from __future__ import annotations

import numpy as np
import pandas as pd

from .io import COMPONENTS, STATES, season_of

PM_MONTH_DAYS = [(3, 15), (7, 15), (12, 15)]
# Weibull fit of bearing-position renewal intervals on the operating-hour clock, EXCLUDING the sealed
# holdout conveyors (scripts/02_reliability_analysis.py Â§A2 -> outputs/req1/weibull_bearing_ml_constants.csv):
# common beta, eta by load class. Must stay fit on non-sealed conveyors only; see evaluate.sealed_conveyors.
BRG_BETA = 2.97
BRG_ETA_OH = {0: 45885.6, 1: 45885.6 * 0.162, 2: 45885.6 * 0.079}  # Light, Medium, Heavy
HORIZON_DAYS = 1096  # 3 years
CM_SIGNALS = ["vib", "cur_per_tp", "vdrop_per_cur", "mtemp_excess"]


def _last_idx(mask: np.ndarray) -> np.ndarray:
    """Index of the most recent True at or before each position, or -1."""
    return np.maximum.accumulate(np.where(mask, np.arange(len(mask)), -1))


def _roll_sum(x: np.ndarray, w: int) -> np.ndarray:
    cs = np.concatenate([[0.0], np.cumsum(x, dtype=np.float64)])
    i = np.arange(1, len(x) + 1)
    return cs[i] - cs[np.maximum(i - w, 0)]


def _roll_mean_nan(x: np.ndarray, w: int) -> np.ndarray:
    ok = ~np.isnan(x)
    s, c = _roll_sum(np.where(ok, x, 0.0), w), _roll_sum(ok.astype(float), w)
    with np.errstate(invalid="ignore", divide="ignore"):
        return np.where(c > 0, s / c, np.nan)


def _since(ref: np.ndarray, last: np.ndarray, never_value: np.ndarray) -> np.ndarray:
    return np.where(last >= 0, ref - ref[np.maximum(last, 0)], never_value)


def days_to_next_pm(dates: pd.DatetimeIndex) -> np.ndarray:
    years = dates.year.to_numpy()
    day0 = dates.to_numpy().astype("datetime64[D]")
    best = np.full(len(dates), 10_000, dtype=np.int64)
    for dy in (0, 1):
        for m, d in PM_MONTH_DAYS:
            cand = (years + dy - 1970).astype("datetime64[Y]").astype("datetime64[M]") + (m - 1)
            diff = (cand.astype("datetime64[D]") + (d - 1) - day0).astype(np.int64)
            best = np.where((diff >= 1) & (diff < best), diff, best)
    return best


def season_day_counts(start: pd.Timestamp, n_days: int) -> np.ndarray:
    """Cumulative count of days per season from `start`: shape (n_days + 1, 4)."""
    seas = season_of(pd.date_range(start, periods=n_days, freq="D"))
    onehot = np.zeros((n_days, 4))
    onehot[np.arange(n_days), seas] = 1
    return np.vstack([np.zeros(4), np.cumsum(onehot, axis=0)])


def daily_features(df: pd.DataFrame, fleet_tmax: list[float]) -> pd.DataFrame:
    """Features for every day of one cleaned conveyor frame (row i = state at end of day i)."""
    n = len(df)
    i = np.arange(n)
    dates = pd.DatetimeIndex(df["Date"])
    st = df["Daily_State"].to_numpy()
    ft = df["Failure_Type"].fillna("").to_numpy()
    fail = st == "FAILURE_DAY"
    at_risk = (st == "RUNNING") | fail
    age = df["Age_Calendar_Days"].to_numpy(dtype=float)
    cum_oh = df["Cumulative_Operating_Hours"].to_numpy(dtype=float)
    cum_cyc = df["Cumulative_Start_Stop_Cycles"].to_numpy(dtype=float)
    cum_kg = df["Cumulative_kg"].to_numpy(dtype=float)
    f: dict[str, np.ndarray] = {}

    # Static attributes (Plant_ID deliberately excluded: the model must generalise to unseen plants).
    f["load_class"] = df["Load_Class"].map({"Light": 0, "Medium": 1, "Heavy": 2}).to_numpy(dtype=float)
    for c in ["Length_m", "Bearing_Count", "Rated_Throughput_kg_h", "Belt_Speed_m_s", "Roller_Diameter_mm"]:
        f[c] = df[c].to_numpy(dtype=float)

    # Age and exposure.
    f["age_days"], f["cum_oh"], f["cum_cycles"], f["cum_kg"] = age, cum_oh, cum_cyc, cum_kg
    f["availability_all"] = np.cumsum(at_risk) / (i + 1)

    # Overall failure history and recent rates per at-risk day.
    risk_cs = np.cumsum(at_risk)
    f["n_fail_all"] = np.cumsum(fail).astype(float)
    f["rate_all"] = f["n_fail_all"] / np.maximum(risk_cs, 1)
    for w in (30, 90, 365):
        f[f"rate_{w}"] = _roll_sum(fail, w) / np.maximum(_roll_sum(at_risk, w), 1)
    last_fail = _last_idx(fail)
    f["days_since_fail"] = np.where(last_fail >= 0, i - last_fail, age)

    # Per-component clocks since last failure (never failed -> clock since EIS, flagged).
    for comp in COMPONENTS:
        m = fail & (ft == comp)
        last = _last_idx(m)
        k = comp.lower()
        f[f"{k}_n"] = np.cumsum(m).astype(float)
        f[f"{k}_never"] = (last < 0).astype(float)
        f[f"{k}_since_days"] = np.where(last >= 0, i - last, age)
        f[f"{k}_since_oh"] = _since(cum_oh, last, cum_oh)
        f[f"{k}_since_cyc"] = _since(cum_cyc, last, cum_cyc)
        f[f"{k}_since_kg"] = _since(cum_kg, last, cum_kg)
        f[f"{k}_rate_365"] = _roll_sum(m, 365) / np.maximum(_roll_sum(at_risk, 365), 1)
        f[f"{k}_rate_all"] = f[f"{k}_n"] / np.maximum(risk_cs, 1)

    # Bearing population: per-position age in operating hours since its last renewal.
    B = int(df["Bearing_Count"].iloc[0])
    last_b = np.full((n, B), -1, dtype=np.int64)
    ids = df["Failed_Component_ID"].fillna("").to_numpy()
    bmask = fail & (ft == "Bearing") & (pd.Series(ids).str.startswith("BRG_").to_numpy())
    rows = np.flatnonzero(bmask)
    pos = np.array([int(s[4:]) - 1 for s in ids[rows]], dtype=np.int64)
    keep = (pos >= 0) & (pos < B)
    last_b[rows[keep], pos[keep]] = rows[keep]
    last_b = np.maximum.accumulate(last_b, axis=0)
    age_oh = np.where(last_b >= 0, cum_oh[:, None] - cum_oh[np.maximum(last_b, 0)], cum_oh[:, None])
    f["brg_never"] = (last_b < 0).sum(axis=1).astype(float)
    f["brg_age_mean"] = age_oh.mean(axis=1)
    for q in (0, 25, 50, 75, 90, 100):
        f[f"brg_age_p{q}"] = np.percentile(age_oh, q, axis=1)
    f["brg_age_std"] = age_oh.std(axis=1)
    # Expected bearing failures over the next d days: sum over positions of the Weibull cumulative-hazard
    # increment (first failure per position), at the conveyor's recent operating hours per day.
    eta = BRG_ETA_OH.get(int(np.nan_to_num(f["load_class"][0], nan=1)), BRG_ETA_OH[1])
    oh_per_day = np.maximum(_roll_sum(np.diff(cum_oh, prepend=cum_oh[0]), 90) / np.minimum(i + 1, 90), 1.0)
    h_now = (age_oh / eta) ** BRG_BETA
    for d in (7, 30, 90, 365):
        dh = (oh_per_day * d)[:, None]
        f[f"brg_whaz_next{d}"] = (((age_oh + dh) / eta) ** BRG_BETA - h_now).sum(axis=1)

    # Restart / state at origin / planned maintenance / calendar.
    down = ~at_risk
    f["days_since_restart"] = np.where(_last_idx(down) >= 0, i - _last_idx(down), age)
    f["state_code"] = pd.Series(st).map({s: k for k, s in enumerate(STATES)}).to_numpy(dtype=float)
    pm = st == "PLANNED_MAINTENANCE"
    f["days_since_pm"] = np.where(_last_idx(pm) >= 0, i - _last_idx(pm), age)
    f["days_to_next_pm"] = days_to_next_pm(dates).astype(float)
    f["doy"] = dates.dayofyear.to_numpy(dtype=float)
    f["season"] = season_of(dates).astype(float)

    # Climatology: own per-season temperature levels (expanding mean), humidity and voltage levels.
    seas = season_of(dates)
    tmax = df["Temperature_Max_C"].to_numpy(dtype=float)
    ok = ~np.isnan(tmax)
    levels = np.empty((n, 4))
    for s in range(4):
        m = ok & (seas == s)
        num, den = np.cumsum(np.where(m, tmax, 0.0)), np.cumsum(m)
        with np.errstate(invalid="ignore", divide="ignore"):
            levels[:, s] = np.where(den > 0, num / den, np.nan)
    resid = np.where(ok, tmax - np.asarray(fleet_tmax)[seas], 0.0)
    offset = np.cumsum(resid) / np.maximum(np.cumsum(ok), 1)
    for s in range(4):  # unseen season so far: fleet level + this conveyor's mean offset
        levels[:, s] = np.where(np.isnan(levels[:, s]), fleet_tmax[s] + offset, levels[:, s])
        f[f"temp_level_s{s}"] = levels[:, s]
    f["temp_offset"] = offset
    cal = season_day_counts(dates[0], n + HORIZON_DAYS + 2)
    for w in (30, 90, 365):
        counts = cal[i + w + 1] - cal[i + 1]
        f[f"temp_exp_next{w}"] = (counts * levels).sum(axis=1) / w
    for col, name in [("Humidity_pct", "humidity_level"), ("Voltage_V", "voltage_level")]:
        x = df[col].to_numpy(dtype=float)
        okx = ~np.isnan(x)
        f[name] = np.cumsum(np.where(okx, x, 0.0)) / np.maximum(np.cumsum(okx), 1)

    # Condition monitoring at origin: 7/30-day means over operating days, and 30-day change.
    tp = df["Throughput_kg_per_h"].to_numpy(dtype=float)
    cur = df["Motor_Current_A"].to_numpy(dtype=float)
    with np.errstate(invalid="ignore", divide="ignore"):
        cm = {
            "vib": df["Structure_Vibration_RMS_mm_s"].to_numpy(dtype=float),
            "cur_per_tp": cur / tp * 1000.0,
            "vdrop_per_cur": df["Contact_Voltage_Drop_mV"].to_numpy(dtype=float) / cur,
            "mtemp_excess": df["Motor_Temperature_C"].to_numpy(dtype=float) - tmax,
        }
    for k, x in cm.items():
        m7, m30 = _roll_mean_nan(x, 7), _roll_mean_nan(x, 30)
        f[f"cm_{k}_7"], f[f"cm_{k}_30"] = m7, m30
        f[f"cm_{k}_d30"] = m30 - np.concatenate([np.full(30, np.nan), m30[:-30]])
    close = df["Contactor_Closing_Time_ms"].to_numpy(dtype=float)
    cs = pd.Series(close).dropna().rolling(5, min_periods=1).mean()
    f["cm_closing_last5"] = pd.Series(np.nan, index=range(n)).fillna(cs).ffill().to_numpy()
    f["tp_90"] = _roll_mean_nan(tp, 90)

    out = pd.DataFrame(f).astype(np.float32)
    out.insert(0, "Date", dates)
    out.insert(0, "Conveyor_ID", df["Conveyor_ID"].iloc[0])
    out.insert(2, "Plant_ID", df["Plant_ID"].iloc[0])
    return out


def features_asof(df: pd.DataFrame, fleet_tmax: list[float]) -> pd.DataFrame:
    """Features at the last observed day (the forecast origin)."""
    return daily_features(df, fleet_tmax).iloc[[-1]].reset_index(drop=True)


In [ ]:
%%writefile conveyor/labels.py
"""Future labels for training origins (training only: these look past the origin by design).

For origin row i (end of day i), with T = last observed row index:
- t{k}, e{k}: days from origin to the k-th next failure (k=1..5), event flag (0 = right-censored at T-i).
- c{k}: component code of the k-th next failure (-1 if censored).
- nx_{comp}, ex_{comp}: days to the next failure of each component, event flag.
- m{j}_fail, m{j}_dt, m{j}_pm: failures, downtime hours, PM days in horizon block j=1..36
  (block j covers days (edge[j-1], edge[j]] after the origin); NaN when the block passes T.
"""
from __future__ import annotations

import numpy as np
import pandas as pd

from .features import HORIZON_DAYS
from .io import COMPONENTS

N_NEXT = 5
N_BLOCKS = 36
BLOCK_EDGES = np.round(np.arange(N_BLOCKS + 1) * HORIZON_DAYS / N_BLOCKS).astype(int)  # 0..1096


def _next_after(event_idx: np.ndarray, origins: np.ndarray, k: int, T: int):
    """Days to the k-th event strictly after each origin, and event flags (censored at T)."""
    j = np.searchsorted(event_idx, origins, side="right") + (k - 1)
    has = j < len(event_idx)
    padded = np.append(event_idx, T)  # censored rows index the sentinel T
    t = np.where(has, padded[np.minimum(j, len(event_idx))], T) - origins
    return t.astype(np.float32), has.astype(np.int8), np.where(has, j, -1)


def labels_for(df: pd.DataFrame, origins: np.ndarray) -> pd.DataFrame:
    T = len(df) - 1
    st = df["Daily_State"].to_numpy()
    ft = df["Failure_Type"].fillna("").to_numpy()
    fail_idx = np.flatnonzero(st == "FAILURE_DAY")
    comp_code = pd.Series(ft[fail_idx]).map({c: k for k, c in enumerate(COMPONENTS)}).fillna(-1).to_numpy()
    out = {"Conveyor_ID": df["Conveyor_ID"].iloc[0], "Date": df["Date"].to_numpy()[origins]}

    for k in range(1, N_NEXT + 1):
        t, e, j = _next_after(fail_idx, origins, k, T)
        out[f"t{k}"], out[f"e{k}"] = t, e
        out[f"c{k}"] = np.append(comp_code, -1)[np.where(j >= 0, j, len(comp_code))].astype(np.int8)
    for comp in COMPONENTS:
        idx = fail_idx[ft[fail_idx] == comp]
        t, e, _ = _next_after(idx, origins, 1, T)
        out[f"nx_{comp.lower()}"], out[f"ex_{comp.lower()}"] = t, e

    cs_fail = np.concatenate([[0], np.cumsum(st == "FAILURE_DAY")])
    cs_dt = np.concatenate([[0.0], np.cumsum(df["Downtime_Hours_Day"].fillna(0).to_numpy(dtype=float))])
    cs_pm = np.concatenate([[0], np.cumsum(st == "PLANNED_MAINTENANCE")])

    def block(cs, a, b):  # sum over days (origin+a, origin+b]; NaN if past the last row
        hi = origins + b
        ok = hi <= T
        v = cs[np.minimum(hi, T) + 1] - cs[np.minimum(origins + a, T) + 1]
        return np.where(ok, v, np.nan).astype(np.float32)

    for jb in range(1, N_BLOCKS + 1):
        a, b = BLOCK_EDGES[jb - 1], BLOCK_EDGES[jb]
        out[f"m{jb}_fail"] = block(cs_fail, a, b)
        out[f"m{jb}_dt"] = block(cs_dt, a, b)
        out[f"m{jb}_pm"] = block(cs_pm, a, b)
    out["tot_fail"] = block(cs_fail, 0, HORIZON_DAYS)
    out["tot_dt"] = block(cs_dt, 0, HORIZON_DAYS)
    out["days_observed_after"] = (T - origins).astype(np.int32)
    return pd.DataFrame(out)


In [ ]:
%%writefile conveyor/evaluate.py
"""Grouped folds, known-future helpers and conveyor-weighted metrics for Req. 2 and Req. 3."""
from __future__ import annotations

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

from .features import PM_MONTH_DAYS
from .labels import BLOCK_EDGES, N_BLOCKS, N_NEXT

SEALED_FRACTION = 0.15
ALWAYS_SEALED = ["P02CV27"]  # example conveyor: keeps the example-file demo leak-free


def sealed_conveyors(conv_plant: pd.Series, seed: int = 0) -> list[str]:
    """~15% of conveyors per plant, fixed forever (touched once, at the very end)."""
    rng = np.random.default_rng(seed)
    out = []
    for _, ids in conv_plant.groupby(conv_plant).groups.items():
        forced = [i for i in ALWAYS_SEALED if i in ids]
        rest = sorted(set(ids) - set(forced))
        k = max(1, round(SEALED_FRACTION * len(ids))) - len(forced)
        out += forced + list(rng.choice(rest, size=max(k, 0), replace=False))
    return sorted(out)


def cv_folds(conv_plant: pd.Series, n_splits: int = 5, seed: int = 0) -> list[tuple[list[str], list[str]]]:
    """(train_ids, test_ids) per fold over non-sealed conveyors; test folds are whole conveyors, stratified by plant."""
    ids = conv_plant.index.to_numpy()
    sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    return [(list(ids[tr]), list(ids[te])) for tr, te in sgkf.split(ids, conv_plant.to_numpy(), groups=ids)]


def nominal_pm_blocks(origin_dates) -> np.ndarray:
    """Count of nominal PM dates (03-15, 07-15, 12-15) in each horizon block after each origin: (n, 36)."""
    d0 = pd.DatetimeIndex(origin_dates).to_numpy().astype("datetime64[D]")
    years = pd.DatetimeIndex(origin_dates).year.to_numpy()
    out = np.zeros((len(d0), N_BLOCKS), dtype=np.float32)
    for dy in range(0, 5):
        for m, d in PM_MONTH_DAYS:
            pm = (years + dy - 1970).astype("datetime64[Y]").astype("datetime64[M]") + (m - 1)
            off = (pm.astype("datetime64[D]") + (d - 1) - d0).astype(np.int64)
            blk = np.searchsorted(BLOCK_EDGES, off, side="left") - 1  # off in (edge[b], edge[b+1]]
            ok = (off >= 1) & (off <= BLOCK_EDGES[-1])
            np.add.at(out, (np.flatnonzero(ok), blk[ok]), 1)
    return out


def _conv_mean(values: np.ndarray, conv: np.ndarray, mask: np.ndarray) -> float:
    """Average per conveyor first, then across conveyors (so Heavy conveyors don't dominate)."""
    if mask.sum() == 0:
        return float("nan")
    s = pd.Series(values[mask]).groupby(conv[mask]).mean()
    return float(s.mean())


def score(pred: dict, Y: pd.DataFrame) -> dict:
    """pred keys: t1..t5 (days, point), c1..c5 (component code), fail_blocks (n,36), pm_blocks (n,36)."""
    conv = Y["Conveyor_ID"].to_numpy()
    res = {}
    for k in range(1, N_NEXT + 1):
        ev = Y[f"e{k}"].to_numpy() == 1
        true_t = Y[f"t{k}"].to_numpy()
        if f"t{k}" in pred:
            p = np.asarray(pred[f"t{k}"], dtype=float)
            res[f"t{k}_logerr"] = _conv_mean(np.abs(np.log1p(p) - np.log1p(true_t)), conv, ev)
            res[f"t{k}_mae"] = _conv_mean(np.abs(p - true_t), conv, ev)
        if f"c{k}" in pred:
            res[f"c{k}_acc"] = _conv_mean((np.asarray(pred[f"c{k}"]) == Y[f"c{k}"].to_numpy()).astype(float), conv, ev)
    if "fail_blocks" in pred:
        fb = np.asarray(pred["fail_blocks"], dtype=float)
        pmb = np.asarray(pred.get("pm_blocks", nominal_pm_blocks(Y["Date"])), dtype=float)
        dt_pred = 36.0 * fb + 24.0 * pmb
        true_dt = Y[[f"m{j}_dt" for j in range(1, N_BLOCKS + 1)]].to_numpy(dtype=float)
        true_f = Y[[f"m{j}_fail" for j in range(1, N_BLOCKS + 1)]].to_numpy(dtype=float)
        complete = ~np.isnan(true_dt).any(axis=1)
        tot_p, tot_t = dt_pred.sum(axis=1), true_dt.sum(axis=1)
        res["dt_tot_abs"] = _conv_mean(np.abs(tot_p - tot_t), conv, complete)
        res["dt_tot_pct"] = _conv_mean(np.abs(tot_p - tot_t) / np.maximum(tot_t, 1.0), conv, complete)
        res["dt_tot_bias"] = _conv_mean(tot_p - tot_t, conv, complete)
        sq = (dt_pred - true_dt) ** 2
        n_obs = (~np.isnan(sq)).sum(axis=1)
        rmse = np.sqrt(np.nansum(sq, axis=1) / np.maximum(n_obs, 1))
        res["dt_month_rmse"] = _conv_mean(rmse, conv, n_obs > 0)
        res["fail_tot_pct"] = _conv_mean(np.abs(fb.sum(1) - true_f.sum(1)) / np.maximum(true_f.sum(1), 1), conv, complete)
        if "tot_dt_lo" in pred:
            inside = (tot_t >= pred["tot_dt_lo"]) & (tot_t <= pred["tot_dt_hi"])
            res["dt_cover80"] = _conv_mean(inside.astype(float), conv, complete)
    return res


def score_by(pred: dict, Y: pd.DataFrame, X: pd.DataFrame) -> pd.DataFrame:
    """Scores overall, by load class, and by origin age bucket."""
    rows = {"all": score(pred, Y)}
    groups = {
        "Light": X["load_class"].to_numpy() == 0, "Medium": X["load_class"].to_numpy() == 1,
        "Heavy": X["load_class"].to_numpy() == 2,
        "age<1y": X["age_days"].to_numpy() < 365, "age1-3y": (X["age_days"].to_numpy() >= 365) & (X["age_days"].to_numpy() < 1095),
        "age3y+": X["age_days"].to_numpy() >= 1095,
    }
    for g, m in groups.items():
        sub = {k: (v[m] if isinstance(v, np.ndarray) and len(v) == len(m) else v) for k, v in pred.items()}
        rows[g] = score(sub, Y.loc[m].reset_index(drop=True))
    return pd.DataFrame(rows).T


In [ ]:
%%writefile conveyor/models.py
"""Forecasting models. Each has fit(X, Y) and predict(X) -> pred dict (see evaluate.score).

- Baselines: OwnRate (conveyor's own failure rate) and EmpiricalBayes (shrunk to the load-class prior).
- DirectML: XGBoost AFT for days to the k-th next failure (censoring-aware), LightGBM multiclass for
  the k-th failure's component, and a stacked LightGBM Poisson model for failures per horizon block.
"""
from __future__ import annotations

import os

import lightgbm as lgb
import numpy as np
import pandas as pd
import xgboost as xgb
from scipy.stats import nbinom

from .evaluate import nominal_pm_blocks
from .features import season_day_counts
from .labels import BLOCK_EDGES, N_BLOCKS, N_NEXT

ID_COLS = ["Conveyor_ID", "Date", "Plant_ID"]
BLOCK_LEN = np.diff(BLOCK_EDGES).astype(np.float32)
CAL_START = pd.Timestamp("2000-01-01")
CAL = season_day_counts(CAL_START, 365 * 60)  # cumulative season-day counts, 2000..2060


def feature_cols(X: pd.DataFrame, drop: tuple[str, ...] = ()) -> list[str]:
    return [c for c in X.columns if c not in ID_COLS and not any(c.startswith(p) for p in drop)]


def block_features(X: pd.DataFrame) -> dict[str, np.ndarray]:
    """Known-future features per horizon block: (n, 36) arrays."""
    base = (pd.DatetimeIndex(X["Date"]) - CAL_START).days.to_numpy()
    levels = X[[f"temp_level_s{s}" for s in range(4)]].to_numpy(dtype=float)
    temp = np.empty((len(X), N_BLOCKS), dtype=np.float32)
    for j in range(N_BLOCKS):
        counts = CAL[base + BLOCK_EDGES[j + 1] + 1] - CAL[base + BLOCK_EDGES[j] + 1]
        temp[:, j] = (counts * levels).sum(axis=1) / BLOCK_LEN[j]
    mid = pd.DatetimeIndex(X["Date"]).to_numpy()[:, None] + ((BLOCK_EDGES[:-1] + BLOCK_EDGES[1:]) // 2).astype("timedelta64[D]")
    month = pd.DatetimeIndex(mid.ravel()).month.to_numpy().reshape(len(X), N_BLOCKS).astype(np.float32)
    return {"blk_temp": temp, "blk_month": month, "blk_pm": nominal_pm_blocks(X["Date"])}


def rate_to_pred(p: np.ndarray, X: pd.DataFrame) -> dict:
    """Turn a per-at-risk-day failure probability into Req. 2/3 predictions (geometric process)."""
    p = np.clip(p, 1e-5, 0.95)
    pm = nominal_pm_blocks(X["Date"])
    fail_blocks = (BLOCK_LEN[None, :] - pm) * (p / (1 + p))[:, None]
    extra = (X["state_code"].to_numpy() == 1).astype(float)  # origin is a failure day -> next day corrective
    pred = {"fail_blocks": fail_blocks, "pm_blocks": pm}
    for k in range(1, N_NEXT + 1):
        at_risk_days = nbinom.median(k, p) + k
        pred[f"t{k}"] = at_risk_days + (k - 1) + extra
        pred[f"c{k}"] = np.zeros(len(X), dtype=int)  # Bearing
    return pred


class OwnRate:
    name = "own_rate"

    def fit(self, X, Y):
        return self

    def predict(self, X):
        return rate_to_pred(X["rate_all"].to_numpy(dtype=float), X)


class EmpiricalBayes:
    """Recent own rate shrunk toward the load-class prior (prior from mature training conveyors)."""

    def __init__(self, a: float = 200.0):
        self.a, self.name = a, f"emp_bayes_a{int(a)}"

    def fit(self, X, Y):
        mature = X["age_days"] >= 730
        self.prior = X[mature].groupby("load_class")["rate_365"].mean().to_dict()
        return self

    def predict(self, X):
        prior = X["load_class"].map(self.prior).to_numpy(dtype=float)
        h = np.minimum(X["age_days"].to_numpy() + 1, 365) * X["availability_all"].to_numpy()
        p = (X["rate_365"].to_numpy() * h + self.a * prior) / (h + self.a)
        return rate_to_pred(p, X)


LGB_COUNT = dict(objective="poisson", learning_rate=0.05, num_leaves=63, min_data_in_leaf=200,
                 feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1, lambda_l2=1.0, verbose=-1)
LGB_COMP = dict(objective="multiclass", num_class=7, learning_rate=0.05, num_leaves=31, min_data_in_leaf=200,
                feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1, lambda_l2=1.0, verbose=-1)
XGB_AFT = dict(objective="survival:aft", eval_metric="aft-nloglik", aft_loss_distribution="normal",
               aft_loss_distribution_scale=1.0, tree_method="hist", learning_rate=0.05, max_depth=6,
               min_child_weight=20, subsample=0.8, colsample_bytree=0.8, nthread=-1,
               device=os.environ.get("CONVEYOR_XGB_DEVICE", "cpu"))  # set to "cuda" on a GPU machine


class DirectML:
    name = "direct_ml"

    def __init__(self, drop: tuple[str, ...] = (), count_stride: int = 4, rounds_count: int = 600,
                 rounds_aft: int = 400, rounds_comp: int = 300, aft_dist: str = "normal", seed: int = 0,
                 lgb_count: dict | None = None, xgb_aft: dict | None = None, lgb_comp: dict | None = None):
        self.drop, self.count_stride, self.seed = drop, count_stride, seed
        self.rounds = dict(count=rounds_count, aft=rounds_aft, comp=rounds_comp)
        self.p_count = {**LGB_COUNT, **(lgb_count or {}), "seed": seed}
        self.p_aft = {**XGB_AFT, **(xgb_aft or {}), "aft_loss_distribution": aft_dist, "seed": seed}
        self.p_comp = {**LGB_COMP, **(lgb_comp or {}), "seed": seed}
        self.name = "direct_ml" + ("_drop-" + "-".join(drop) if drop else "")

    # --- stacked count model -------------------------------------------------------------
    def _long(self, X: pd.DataFrame, Xf: np.ndarray):
        bf = block_features(X)
        n = len(X)
        rep = np.repeat(np.arange(n), N_BLOCKS)
        j = np.tile(np.arange(N_BLOCKS), n)
        extra = np.column_stack([j + 1, BLOCK_LEN[j], bf["blk_temp"].ravel(), bf["blk_month"].ravel(), bf["blk_pm"].ravel()])
        return np.hstack([Xf[rep], extra.astype(np.float32)])

    def fit(self, X: pd.DataFrame, Y: pd.DataFrame):
        self.cols = feature_cols(X, self.drop)
        Xf = X[self.cols].to_numpy(dtype=np.float32)
        w = 1.0 / X.groupby("Conveyor_ID")["Conveyor_ID"].transform("size").to_numpy()  # equal weight per conveyor
        w = w / w.mean()

        # Req. 3: failures per block (stacked over blocks), subsampled origins.
        sub = np.arange(0, len(X), self.count_stride)
        L = self._long(X.iloc[sub], Xf[sub])
        yb = Y.iloc[sub][[f"m{j}_fail" for j in range(1, N_BLOCKS + 1)]].to_numpy(dtype=np.float32).ravel()
        wb = np.repeat(w[sub], N_BLOCKS)
        ok = ~np.isnan(yb)
        self.count_model = lgb.train(self.p_count, lgb.Dataset(L[ok], yb[ok], weight=wb[ok]), self.rounds["count"])

        # Req. 2: days to k-th failure (AFT, censoring-aware) and its component (multiclass).
        self.aft, self.comp = [], []
        for k in range(1, N_NEXT + 1):
            t = Y[f"t{k}"].to_numpy(dtype=float)
            ev = Y[f"e{k}"].to_numpy() == 1
            lo = np.maximum(t, 0.5)
            hi = np.where(ev, lo, np.inf)
            d = xgb.DMatrix(Xf, weight=w)
            d.set_float_info("label_lower_bound", lo)
            d.set_float_info("label_upper_bound", hi)
            self.aft.append(xgb.train(self.p_aft, d, self.rounds["aft"]))
            c = Y[f"c{k}"].to_numpy()
            self.comp.append(lgb.train(self.p_comp, lgb.Dataset(Xf[ev], c[ev], weight=w[ev]), self.rounds["comp"]))
        return self

    # --- persistence: text/JSON only, so artifacts don't depend on pickle or library versions ---
    def save(self, d):
        import json
        from pathlib import Path
        d = Path(d)
        d.mkdir(parents=True, exist_ok=True)
        self.count_model.save_model(str(d / "count.txt"))
        for k in range(N_NEXT):
            self.aft[k].save_model(str(d / f"aft_{k+1}.json"))
            self.comp[k].save_model(str(d / f"comp_{k+1}.txt"))
        (d / "meta.json").write_text(json.dumps({"cols": self.cols, "name": self.name}, indent=1), newline="\n")

    @classmethod
    def load(cls, d):
        import json
        from pathlib import Path
        d = Path(d)
        m = cls.__new__(cls)
        meta = json.loads((d / "meta.json").read_text())
        m.cols, m.name = meta["cols"], meta["name"]
        m.count_model = lgb.Booster(model_file=str(d / "count.txt"))
        m.aft, m.comp = [], []
        for k in range(N_NEXT):
            b = xgb.Booster()
            b.load_model(str(d / f"aft_{k+1}.json"))
            b.set_param({"device": "cpu"})  # models may be trained on a GPU; inference (Colab) runs on CPU
            m.aft.append(b)
            m.comp.append(lgb.Booster(model_file=str(d / f"comp_{k+1}.txt")))
        return m

    def predict(self, X: pd.DataFrame) -> dict:
        Xf = X[self.cols].to_numpy(dtype=np.float32)
        fb = self.count_model.predict(self._long(X, Xf)).reshape(len(X), N_BLOCKS)
        pred = {"fail_blocks": fb, "pm_blocks": nominal_pm_blocks(X["Date"])}
        d = xgb.DMatrix(Xf)
        prev = np.zeros(len(X))
        for k in range(N_NEXT):
            t = self.aft[k].predict(d)
            t = np.maximum(t, np.where(prev > 0, prev + 2, 1))  # ordered; a failure day is followed by a corrective day
            pred[f"t{k+1}"], prev = t, t
            proba = self.comp[k].predict(Xf)
            pred[f"c{k+1}"], pred[f"p{k+1}"] = proba.argmax(axis=1), proba
        return pred


In [ ]:
%%writefile conveyor/train.py
"""Training workflow: fleet Parquet -> training tables -> calibration -> final bagged models -> artifacts/.

This is the code that produced the submitted artifacts. scripts/01, 03 and 05 call it offline, and the
notebook can run it end to end (`train_all`). Hyperparameters come from the Optuna study (scripts/04_tune.py),
stored in artifacts/tuned/*.json.
"""
from __future__ import annotations

import hashlib
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd

from .evaluate import cv_folds, sealed_conveyors
from .features import daily_features
from .io import iter_conveyors, load_fleet, season_of
from .labels import N_BLOCKS, N_NEXT, labels_for
from .models import DirectML, EmpiricalBayes, OwnRate

MIN_HISTORY = 200  # first training origin at index 199 = 200 days of history (the spec's minimum input)
STEP = 7  # weekly training origins
LOADS = {0: "Light", 1: "Medium", 2: "Heavy"}


def _log(msg: str) -> None:
    print(msg, flush=True)


DROPS = {"direct": (), "direct-nocm": ("cm_",), "direct-noclim": ("temp_", "humidity", "voltage"),
         "direct-nobrg": ("brg_",)}


def fit_climatology(fleet: pd.DataFrame) -> list[float]:
    """Fleet mean Tmax per season (DJF, MAM, JJA, SON), fit on non-sealed conveyors only (it feeds the model)."""
    conv_plant = fleet.groupby("Conveyor_ID")["Plant_ID"].first()
    clim_fleet = fleet[~fleet["Conveyor_ID"].isin(sealed_conveyors(conv_plant))]
    seas = season_of(clim_fleet["Date"])
    return [float(clim_fleet["Temperature_Max_C"][seas == s].mean()) for s in range(4)]


def build_tables(fleet: pd.DataFrame, fleet_tmax, log=_log) -> tuple[pd.DataFrame, pd.DataFrame]:
    """As-of features (X) and future labels (Y) at weekly origins from day 200, for every conveyor."""
    t0 = time.time()
    feats, labs = [], []
    for cid, df in iter_conveyors(fleet):
        origins = np.arange(MIN_HISTORY - 1, len(df) - 1, STEP)
        feats.append(daily_features(df, fleet_tmax).iloc[origins].reset_index(drop=True))
        labs.append(labels_for(df, origins))
    X = pd.concat(feats, ignore_index=True)
    Y = pd.concat(labs, ignore_index=True)
    assert (X["Conveyor_ID"].to_numpy() == Y["Conveyor_ID"].to_numpy()).all()
    assert (X["Date"].to_numpy() == Y["Date"].to_numpy()).all()
    log(f"features {X.shape}, labels {Y.shape}, built in {time.time()-t0:.0f}s")
    return X, Y


def tuned_kwargs(tuned_dir: str | Path = "artifacts/tuned") -> dict:
    """DirectML keyword arguments from the Optuna best params (defaults where a study is missing)."""
    kw = {}
    for sub, (pkey, rkey) in {"count": ("lgb_count", "rounds_count"), "aft": ("xgb_aft", "rounds_aft"),
                              "comp": ("lgb_comp", "rounds_comp")}.items():
        f = Path(tuned_dir) / f"{sub}.json"
        if f.exists():
            t = json.loads(f.read_text())
            kw[pkey], kw[rkey] = dict(t["params"]), int(t["rounds"])
    if "xgb_aft" in kw:
        kw["aft_dist"] = kw["xgb_aft"].pop("aft_loss_distribution", "normal")
    return kw


def make(spec: str, seed: int, tuned_dir: str | Path = "artifacts/tuned"):
    """Model factory for a spec name (the submitted tool uses `direct-tuned`)."""
    if spec == "own_rate":
        return OwnRate()
    if spec.startswith("eb:"):
        return EmpiricalBayes(float(spec.split(":")[1]))
    if spec in DROPS:
        m = DirectML(drop=DROPS[spec], seed=seed)
    elif spec == "direct-tuned":
        m = DirectML(seed=seed, **tuned_kwargs(tuned_dir))
    elif spec.startswith("direct-aft:"):
        m = DirectML(aft_dist=spec.split(":")[1], seed=seed)
    else:
        raise ValueError(spec)
    m.name = spec
    return m


def oof_predict(spec: str, X: pd.DataFrame, Y: pd.DataFrame, folds, seed: int = 0,
                tuned_dir: str | Path = "artifacts/tuned", log=_log) -> dict:
    """Out-of-fold predictions: each fold's test conveyors are predicted by a model that never saw them."""
    t0 = time.time()
    oof = None
    for k, (tr, te) in enumerate(folds):
        itr, ite = X["Conveyor_ID"].isin(tr).to_numpy(), X["Conveyor_ID"].isin(te).to_numpy()
        model = make(spec, seed, tuned_dir).fit(X[itr].reset_index(drop=True), Y[itr].reset_index(drop=True))
        pred = model.predict(X[ite].reset_index(drop=True))
        if oof is None:
            oof = {key: np.zeros((len(X),) + np.shape(v)[1:], dtype=np.asarray(v).dtype) for key, v in pred.items()}
        for key, v in pred.items():
            oof[key][ite] = v
        log(f"  fold {k+1}/{len(folds)} done ({time.time()-t0:.0f}s)")
    return oof


def calibration(oof: dict, X: pd.DataFrame, Y: pd.DataFrame) -> dict:
    """Empirical (conformal) q10/q90 of OOF residuals per load class: log(true/pred) of days to the k-th
    failure, and true/pred of the 3-year total downtime. These become the tool's P10/P90 bands."""
    load = X["load_class"].map(LOADS).to_numpy()
    cal = {"t_logratio_q10_q90": {}, "tot_ratio_q10_q90": {}}
    true_dt = Y[[f"m{j}_dt" for j in range(1, N_BLOCKS + 1)]].to_numpy(dtype=float)
    complete = ~np.isnan(true_dt).any(axis=1)
    pred_tot = (36 * oof["fail_blocks"] + 24 * oof["pm_blocks"]).sum(axis=1)
    for lc in LOADS.values():
        m = load == lc
        cal["t_logratio_q10_q90"][lc] = {}
        for k in range(1, N_NEXT + 1):
            ev = m & (Y[f"e{k}"].to_numpy() == 1)
            r = np.log(np.maximum(Y[f"t{k}"].to_numpy()[ev], 1) / oof[f"t{k}"][ev])
            cal["t_logratio_q10_q90"][lc][str(k)] = [float(np.quantile(r, 0.1)), float(np.quantile(r, 0.9))]
        mc = m & complete
        ratio = true_dt.sum(axis=1)[mc] / pred_tot[mc]
        cal["tot_ratio_q10_q90"][lc] = [float(np.quantile(ratio, 0.1)), float(np.quantile(ratio, 0.9))]
    return cal


def fit_final(spec: str, X: pd.DataFrame, Y: pd.DataFrame, seeds: int, art: str | Path,
              tuned_dir: str | Path = "artifacts/tuned", log=_log) -> list[str]:
    """Fit one model per seed on all rows supplied and save each to art/model_seed<s>/ (text/JSON only)."""
    t0 = time.time()
    dirs = []
    for s in range(seeds):
        make(spec, s, tuned_dir).fit(X, Y).save(Path(art) / f"model_seed{s}")
        dirs.append(f"model_seed{s}")
        log(f"  saved model_seed{s} ({time.time()-t0:.0f}s)")
    return dirs


def write_manifest(art: str | Path, spec: str, dirs: list[str], cal: dict, X: pd.DataFrame) -> dict:
    """manifest.json: model list, calibration, and SHA-256 of every artifact file."""
    art = Path(art)
    files = sorted(p for p in art.rglob("*") if p.is_file() and p.name != "manifest.json")
    manifest = {"spec": spec, "model_dirs": dirs, "calibration": cal, "n_conveyors": int(X.Conveyor_ID.nunique()),
                "n_origins": len(X), "created": pd.Timestamp.now().isoformat(timespec="seconds"),
                "sha256": {str(p.relative_to(art)).replace("\\", "/"): hashlib.sha256(p.read_bytes()).hexdigest()
                           for p in files}}
    (art / "manifest.json").write_text(json.dumps(manifest, indent=1), newline="\n")
    return manifest


def train_all(fleet_path: str, art: str | Path = "artifacts", spec: str = "direct-tuned", seeds: int = 5,
              folds: int = 5, tuned_dir: str | Path | None = None, log=_log) -> dict:
    """The full training workflow, from the fleet file to the artifacts the forecast tool loads.

    1. Climatology (non-sealed conveyors) and training tables (weekly origins from day 200, all conveyors).
    2. Calibration: grouped K-fold CV over the non-sealed conveyors -> OOF residual quantiles per load class.
    3. Final models: one per seed on all conveyors -> art/model_seed*/, plus manifest.json.
    """
    art = Path(art)
    tuned_dir = Path(tuned_dir) if tuned_dir is not None else art / "tuned"
    art.mkdir(parents=True, exist_ok=True)
    t0 = time.time()
    fleet = load_fleet(fleet_path)
    fleet_tmax = fit_climatology(fleet)
    (art / "climatology.json").write_text(json.dumps({"fleet_tmax_by_season_DJF_MAM_JJA_SON": fleet_tmax}, indent=1),
                                        newline="\n")  # LF on every OS: the manifest hashes these bytes
    log(f"[1/3] loaded {len(fleet):,} rows ({fleet.Conveyor_ID.nunique()} conveyors) in {time.time()-t0:.0f}s; "
        f"fleet Tmax by season {np.round(fleet_tmax, 2)}")
    X, Y = build_tables(fleet, fleet_tmax, log)
    del fleet

    conv_plant = X.groupby("Conveyor_ID")["Plant_ID"].first()
    sealed = sealed_conveyors(conv_plant)
    keep = ~X["Conveyor_ID"].isin(sealed).to_numpy()
    log(f"[2/3] calibration: {folds}-fold grouped CV over {len(conv_plant) - len(sealed)} conveyors")
    Xc, Yc = X[keep].reset_index(drop=True), Y[keep].reset_index(drop=True)
    oof = oof_predict(spec, Xc, Yc, cv_folds(conv_plant.drop(sealed), folds, 0), 0, tuned_dir, log)
    cal = calibration(oof, Xc, Yc)
    del oof, Xc, Yc

    log(f"[3/3] final fit: {seeds} seeds on all {len(conv_plant)} conveyors")
    dirs = fit_final(spec, X, Y, seeds, art, tuned_dir, log)
    manifest = write_manifest(art, spec, dirs, cal, X)
    log(f"done in {(time.time()-t0)/60:.1f} min; {len(manifest['sha256'])} artifact files")
    return manifest


In [ ]:
%%writefile conveyor/forecast.py
"""The reusable forecasting tool: Parquet path of one conveyor -> next 5 failures + 3-year downtime.

Uses every row supplied, sorts by Date, and forecasts from the last observed day. Nothing about
history length is assumed (minimum 200 days per the spec; shorter input still runs, with a warning).
"""
from __future__ import annotations

import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from .evaluate import nominal_pm_blocks
from .features import features_asof
from .io import COMPONENTS, load_conveyor
from .labels import BLOCK_EDGES, N_BLOCKS, N_NEXT
from .models import DirectML

H_PER_FAILURE = 36.0  # 12 h on the failure day + one 24 h corrective day (constant in the fleet data)
H_PER_PM = 24.0

# Expert rule (fleet observation): speed sensors have a hard design-life ceiling on the operating-hours clock.
# Fleet max life is 50,016 op-h and 158 of 1,838 lives end in 49,980-50,016 (50,000 +/- one half-day).
# Grouped-CV check on held-out conveyors: when the ceiling is < 30 days away, a speed-sensor failure follows
# with probability 0.93 and the date error is within +/-2 days (P10/P90); 30-90 days away: 0.65; further: 0.44.
SS_CEILING_OH = 50_000.0
SS_RULE_CONFIDENCE = [(30, 0.93), (90, 0.65), (np.inf, 0.44)]  # (days-to-ceiling upper bound, P(failure there))
# Lives end at 49,980-50,016 op-h, so a live sensor can already be past 50,000 on the day before it fails (a 50,016
# life is at 50,004 then; 53 of 158 ceiling lives). Up to this overshoot, the failure is due tomorrow. Beyond it, the
# sensor clock is likely wrong (e.g. an unrecorded replacement before the file starts), so the rule stays off.
SS_OVERSHOOT_OH = 24.0


def speed_sensor_ceiling_days(X: pd.DataFrame) -> np.ndarray:
    """Days from the origin until the speed sensor reaches SS_CEILING_OH, at the conveyor's average op-h per day.

    A sensor already past the ceiling by at most SS_OVERSHOOT_OH gets 1 day; further past, <= 0 (rule off).
    """
    oh_per_day = X["cum_oh"].to_numpy(dtype=float) / (X["age_days"].to_numpy(dtype=float) + 1)
    since = X["speed_sensor_since_oh"].to_numpy(dtype=float)
    dc = np.ceil((SS_CEILING_OH - since) / np.maximum(oh_per_day, 1.0))
    return np.where((since >= SS_CEILING_OH) & (since <= SS_CEILING_OH + SS_OVERSHOOT_OH), 1.0, dc)


def apply_expert_rules(pred: dict, X: pd.DataFrame) -> dict:
    """Insert a rule-based Speed_Sensor failure into the next-5 sequence when its ceiling date comes first.

    Later ML failures shift back one slot (and 2 days: the failure day + corrective day). Adds `rule{k}` flags.
    """
    ss = COMPONENTS.index("Speed_Sensor")
    out = {k: np.array(v, copy=True) for k, v in pred.items()}
    T = np.column_stack([out[f"t{k}"] for k in range(1, N_NEXT + 1)]).astype(float)
    P = np.stack([out[f"p{k}"] for k in range(1, N_NEXT + 1)], axis=1)
    R = np.zeros(T.shape, dtype=bool)
    dc = speed_sensor_ceiling_days(X)
    for i in np.flatnonzero((dc >= 1) & (dc < T[:, -1])):
        j = int(np.searchsorted(T[i], dc[i]))
        P[i, :, ss] = 0.0  # the rule now owns the sensor failure; a renewed sensor won't fail again this soon
        P[i] /= P[i].sum(axis=1, keepdims=True)
        conf = next(c for ub, c in SS_RULE_CONFIDENCE if dc[i] < ub)
        p_rule = np.full(P.shape[2], (1 - conf) / (P.shape[2] - 1))
        p_rule[ss] = conf
        T[i] = np.concatenate([T[i, :j], [dc[i]], T[i, j:-1] + 2])
        P[i] = np.concatenate([P[i, :j], p_rule[None], P[i, j:-1]])
        R[i] = np.concatenate([R[i, :j], [True], R[i, j:-1]])
        # Keep >= 2 days between failures (failure day + corrective day). If a failure lands the day before the
        # ceiling date, the conveyor is down that day, accrues no sensor hours, and the sensor fails on restart.
        for k in range(max(j, 1), N_NEXT):
            T[i, k] = max(T[i, k], T[i, k - 1] + 2)
    for k in range(N_NEXT):
        out[f"t{k+1}"], out[f"p{k+1}"], out[f"rule{k+1}"] = T[:, k], P[:, k], R[:, k]
        out[f"c{k+1}"] = P[:, k].argmax(axis=1)
    return out


def load_models(art: Path) -> tuple[list[DirectML], dict]:
    art = Path(art)
    meta = json.loads((art / "manifest.json").read_text())
    models = [DirectML.load(art / d) for d in meta["model_dirs"]]
    return models, meta


def ensemble_predict(models: list[DirectML], X: pd.DataFrame) -> dict:
    preds = [m.predict(X) for m in models]
    out = {"fail_blocks": np.mean([p["fail_blocks"] for p in preds], axis=0),
           "pm_blocks": preds[0]["pm_blocks"]}
    prev = np.zeros(len(X))
    for k in range(1, N_NEXT + 1):
        t = np.exp(np.mean([np.log(p[f"t{k}"]) for p in preds], axis=0))  # geometric mean of medians
        out[f"t{k}"] = prev = np.maximum(t, np.where(prev > 0, prev + 2, 1))
        out[f"p{k}"] = np.mean([p[f"p{k}"] for p in preds], axis=0)
        out[f"c{k}"] = out[f"p{k}"].argmax(axis=1)
    return out


def forecast(path: str, artifacts: str | Path = "artifacts") -> dict:
    """Run the full tool on one conveyor file. Returns dict of DataFrames and a summary."""
    df = load_conveyor(path)
    if len(df) < 200:
        warnings.warn(f"Only {len(df)} days of history (spec guarantees >= 200); forecasting anyway.")
    art = Path(artifacts)
    fleet_tmax = json.loads((art / "climatology.json").read_text())["fleet_tmax_by_season_DJF_MAM_JJA_SON"]
    models, meta = load_models(art)
    cal = meta.get("calibration", {})
    X = features_asof(df, fleet_tmax)
    pred = apply_expert_rules(ensemble_predict(models, X), X)
    last_day = pd.Timestamp(df["Date"].iloc[-1])
    load = str(df["Load_Class"].iloc[0])

    # Req. 2: next five failures.
    rows = []
    n_ml = 0
    for k in range(1, N_NEXT + 1):
        t = float(pred[f"t{k}"][0])
        proba = pred[f"p{k}"][0]
        order = np.argsort(proba)[::-1]
        if pred[f"rule{k}"][0]:
            source = "Rule: speed-sensor 50,000 op-h design-life ceiling"
            lo_d, hi_d = max(1, round(t - max(2.0, 0.3 * t))), round(t + 3)
        else:
            n_ml += 1  # ML slot n_ml keeps the calibration of the model that predicted it
            source = "ML model"
            lo_q, hi_q = cal.get("t_logratio_q10_q90", {}).get(load, {}).get(str(n_ml), [np.nan, np.nan])
            lo_d = max(1, round(t * np.exp(lo_q))) if np.isfinite(lo_q) else None
            hi_d = round(t * np.exp(hi_q)) if np.isfinite(hi_q) else None
        rows.append({
            "Failure #": k,
            "Expected component": COMPONENTS[order[0]],
            "P(component)": round(float(proba[order[0]]), 3),
            "Runner-up": f"{COMPONENTS[order[1]]} ({proba[order[1]]:.2f})",
            "Expected date": (last_day + pd.Timedelta(days=round(t))).date(),
            "Days from last observed day": round(t),
            "P10 date": (last_day + pd.Timedelta(days=lo_d)).date() if lo_d is not None else None,
            "P90 date": (last_day + pd.Timedelta(days=hi_d)).date() if hi_d is not None else None,
            "Source": source,
        })
    next5 = pd.DataFrame(rows)

    # Req. 3: 36 horizon blocks (~1 month each) over the 3 years after the last observed day.
    fb = pred["fail_blocks"][0]
    pm = nominal_pm_blocks([last_day])[0]
    start = [last_day + pd.Timedelta(days=int(BLOCK_EDGES[j]) + 1) for j in range(N_BLOCKS)]
    end = [last_day + pd.Timedelta(days=int(BLOCK_EDGES[j + 1])) for j in range(N_BLOCKS)]
    monthly = pd.DataFrame({
        "Month": np.arange(1, N_BLOCKS + 1), "From": [d.date() for d in start], "To": [d.date() for d in end],
        "Expected failures": fb.round(2),
        "Corrective downtime h": (H_PER_FAILURE * fb).round(1),
        "Planned downtime h": (H_PER_PM * pm).round(1),
    })
    monthly["Total downtime h"] = monthly["Corrective downtime h"] + monthly["Planned downtime h"]
    monthly["Cumulative downtime h"] = monthly["Total downtime h"].cumsum().round(1)
    total = float(monthly["Total downtime h"].sum())
    lo_r, hi_r = cal.get("tot_ratio_q10_q90", {}).get(load, [np.nan, np.nan])
    summary = {
        "conveyor": str(df["Conveyor_ID"].iloc[0]), "load_class": load,
        "history_days": len(df), "first_day": df["Date"].iloc[0].date(), "last_observed_day": last_day.date(),
        "forecast_to": end[-1].date(),
        "expected_failures_3y": round(float(fb.sum()), 1),
        "expected_corrective_downtime_h": round(float(monthly["Corrective downtime h"].sum()), 1),
        "expected_planned_downtime_h": round(float(monthly["Planned downtime h"].sum()), 1),
        "expected_total_downtime_h": round(total, 1),
        "total_downtime_P10_P90_h": (round(total * lo_r, 0), round(total * hi_r, 0)) if np.isfinite(lo_r) else None,
        "speed_sensor_days_to_50k_oh_ceiling": int(speed_sensor_ceiling_days(X)[0]),
    }
    return {"next5": next5, "monthly": monthly, "summary": summary}


def plot_downtime(monthly: pd.DataFrame, title: str = ""):
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(10, 4))
    x = monthly["Month"]
    ax.bar(x, monthly["Corrective downtime h"], label="Corrective (expected)", color="#4C72B0")
    ax.bar(x, monthly["Planned downtime h"], bottom=monthly["Corrective downtime h"], label="Planned", color="#DD8452")
    ax.set_xlabel("Month after last observed day")
    ax.set_ylabel("Downtime h / month")
    ax2 = ax.twinx()
    ax2.plot(x, monthly["Cumulative downtime h"], color="black", lw=2, label="Cumulative")
    ax2.set_ylabel("Cumulative downtime h")
    ax.legend(loc="upper left")
    ax.set_title(title or "Expected downtime over the next 3 years")
    fig.tight_layout()
    return fig


## 4. Models: load the pinned fitted models, or retrain from the fleet file

In [ ]:
def fetch(rel):
    dst = os.path.join('artifacts', rel)
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    urllib.request.urlretrieve(f'{ARTIFACTS_BASE}/artifacts/{rel}', dst)
    return dst

def fetch_verified(rel, sha):
    got = hashlib.sha256(open(fetch(rel), 'rb').read()).hexdigest()
    assert got == sha, f'checksum mismatch: {rel}'

pinned = json.load(open(fetch('manifest.json')))
if not RETRAIN_FROM_FLEET:
    for rel, sha in pinned['sha256'].items():
        fetch_verified(rel, sha)
    print(f"{len(pinned['sha256'])} artifact files verified; models: {pinned['model_dirs']}")
else:
    # Hyperparameters are the Optuna result (appendix); everything else is refit here.
    for rel, sha in pinned['sha256'].items():
        if rel.startswith('tuned/'):
            fetch_verified(rel, sha)
    fleet_path = FLEET_PATH
    if 'drive.google.com' in fleet_path:
        import gdown
        fleet_path = gdown.download(fleet_path, 'fleet.parquet', fuzzy=True, quiet=True)
    elif fleet_path.startswith('http'):
        fleet_path = urllib.request.urlretrieve(fleet_path, 'fleet.parquet')[0]
    from conveyor.train import train_all
    train_all(fleet_path, 'artifacts')

## 5. Run the forecast

In [ ]:
import pandas as pd
from conveyor.forecast import forecast, plot_downtime
pd.set_option('display.width', 200)
if not INPUT_PATH:
    from google.colab import files
    INPUT_PATH = next(iter(files.upload()))
elif INPUT_PATH.startswith('http'):
    INPUT_PATH = urllib.request.urlretrieve(INPUT_PATH, 'input.parquet')[0]
result = forecast(INPUT_PATH, 'artifacts')
for k, v in result['summary'].items():
    print(f'{k:36s} {v}')

### Req. 2: next five failures

In [ ]:
result['next5']

### Req. 3: downtime over the next three years

In [ ]:
s = result['summary']
print(f"Expected total downtime (3 years): {s['expected_total_downtime_h']:.0f} h "
      f"= corrective {s['expected_corrective_downtime_h']:.0f} h + planned {s['expected_planned_downtime_h']:.0f} h; "
      f"80% interval {s['total_downtime_P10_P90_h']}")
fig = plot_downtime(result['monthly'], f"{s['conveyor']}: expected downtime after {s['last_observed_day']}")

In [ ]:
result['monthly']

## Appendix: hyperparameter search (run offline, not executed here)
Optuna studies on 3-fold grouped CV over the non-sealed conveyors; the best parameters are the `artifacts/tuned/*.json` files used by `train.make('direct-tuned')`. It takes hours, so it is written out for the record only (it expects the training tables cached by the offline pipeline).

In [ ]:
%%writefile tune_offline.py
"""Optuna tuning of the DirectML sub-models on grouped CV (3 folds over non-sealed conveyors, seed 0).

One study per sub-model, meant to run as parallel processes (count/comp on CPU, aft on GPU):
- count: LightGBM Poisson on stacked horizon blocks; objective all:dt_tot_pct (3-year total downtime error).
- aft:   XGBoost survival:aft for k=1 and k=5; objective t1_logerr + t5_logerr.
- comp:  LightGBM multiclass for the next failure's component; objective multi_logloss (c1_acc logged).
The number of boosting rounds is picked from the fold-averaged learning curve (a CV choice, not per-fold
early stopping). Trial 0 is the current default, so the best is never worse than it on this harness.
Best params -> artifacts/tuned/<sub>.json, read by the `direct-tuned` spec (conveyor/train.py).

Usage: .venv/Scripts/python scripts/04_tune.py {count,aft,comp} [--timeout 2400] [--threads 8] [--folds 3]
"""
import argparse
import json
import os
import sys
import time
from pathlib import Path

import lightgbm as lgb
import numpy as np
import optuna
import pandas as pd
import xgboost as xgb

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "src"))
from conveyor.evaluate import cv_folds, nominal_pm_blocks, sealed_conveyors  # noqa: E402
from conveyor.labels import N_BLOCKS  # noqa: E402
from conveyor.models import LGB_COMP, LGB_COUNT, XGB_AFT, DirectML, feature_cols  # noqa: E402

CACHE, OUT = Path("outputs/cache"), Path("artifacts/tuned")
TEST_STRIDE = 2  # count study: every 2nd complete test origin (memory); metrics are per-conveyor means anyway


def load(n_folds):
    X = pd.read_parquet(CACHE / "features.parquet")
    Y = pd.read_parquet(CACHE / "labels.parquet")
    conv_plant = X.groupby("Conveyor_ID")["Plant_ID"].first()
    sealed = sealed_conveyors(conv_plant)
    keep = ~X["Conveyor_ID"].isin(sealed).to_numpy()
    X, Y = X[keep].reset_index(drop=True), Y[keep].reset_index(drop=True)
    folds = cv_folds(conv_plant.drop(sealed), n_folds, 0)
    w = 1.0 / X.groupby("Conveyor_ID")["Conveyor_ID"].transform("size").to_numpy()
    return X, Y, folds, w / w.mean()


def conv_mean(err, conv_codes, n_conv):
    s = np.bincount(conv_codes, weights=err, minlength=n_conv)
    c = np.bincount(conv_codes, minlength=n_conv)
    return float((s[c > 0] / c[c > 0]).mean())


# --- count ------------------------------------------------------------------------------------
def prep_count(X, Y, folds, w, stride=4):
    cols = feature_cols(X)
    Xf = X[cols].to_numpy(dtype=np.float32)
    helper = DirectML()
    yb_all = Y[[f"m{j}_fail" for j in range(1, N_BLOCKS + 1)]].to_numpy(dtype=np.float32)
    true_tot = Y[[f"m{j}_dt" for j in range(1, N_BLOCKS + 1)]].to_numpy(dtype=float)
    complete = ~np.isnan(true_tot).any(axis=1)
    conv_codes = pd.factorize(X["Conveyor_ID"])[0]
    data = []
    for tr, te in folds:
        itr = np.flatnonzero(X["Conveyor_ID"].isin(tr).to_numpy())[::stride]
        L = helper._long(X.iloc[itr], Xf[itr])
        yb, wb = yb_all[itr].ravel(), np.repeat(w[itr], N_BLOCKS)
        ok = ~np.isnan(yb)
        dtr = lgb.Dataset(L[ok], yb[ok], weight=wb[ok], params={"feature_pre_filter": False, "verbose": -1}).construct()
        del L
        ite = np.flatnonzero(X["Conveyor_ID"].isin(te).to_numpy() & complete)[::TEST_STRIDE]
        Lte = helper._long(X.iloc[ite], Xf[ite])
        pm_tot = 24.0 * nominal_pm_blocks(X["Date"].iloc[ite]).sum(axis=1)
        dte = lgb.Dataset(Lte, np.nan_to_num(yb_all[ite].ravel()), reference=dtr).construct()
        del Lte
        codes, uniq = pd.factorize(conv_codes[ite])
        data.append(dict(dtr=dtr, dte=dte, n=len(ite), pm_tot=pm_tot, true=true_tot[ite].sum(axis=1),
                         codes=codes, n_conv=len(uniq)))
        print(f"  prepared fold: train {int(ok.sum()):,} block rows, test {len(ite):,} origins", flush=True)
    return data


def count_objective(data, threads):
    def objective(trial):
        lr = trial.suggest_float("learning_rate", 0.015, 0.15, log=True)
        p = {**LGB_COUNT, "learning_rate": lr, "metric": "None", "num_threads": threads, "seed": 0,
             "num_leaves": trial.suggest_int("num_leaves", 15, 255, log=True),
             "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 50, 3000, log=True),
             "feature_fraction": trial.suggest_float("feature_fraction", 0.4, 1.0),
             "bagging_fraction": trial.suggest_float("bagging_fraction", 0.5, 1.0),
             "lambda_l2": trial.suggest_float("lambda_l2", 1e-3, 30, log=True)}
        max_rounds = int(min(2500, 36 / lr))
        curves = []
        for f in data:
            def feval(preds, _ds, f=f):
                tot = 36.0 * preds.reshape(f["n"], N_BLOCKS).sum(axis=1) + f["pm_tot"]
                err = np.abs(tot - f["true"]) / np.maximum(f["true"], 1.0)
                return "dt_tot_pct", conv_mean(err, f["codes"], f["n_conv"]), False
            rec = {}
            lgb.train(p, f["dtr"], max_rounds, valid_sets=[f["dte"]], valid_names=["te"], feval=feval,
                      callbacks=[lgb.record_evaluation(rec)])
            curves.append(rec["te"]["dt_tot_pct"])
        curve = np.mean(curves, axis=0)
        best = int(np.argmin(curve))
        trial.set_user_attr("rounds", best + 1)
        return float(curve[best])
    return objective


def count_default():
    return {k: LGB_COUNT[k] for k in ("learning_rate", "num_leaves", "min_data_in_leaf", "feature_fraction",
                                      "bagging_fraction", "lambda_l2")}


# --- aft --------------------------------------------------------------------------------------
def prep_aft(X, Y, folds, w, ks=(1, 5)):
    Xf = X[feature_cols(X)].to_numpy(dtype=np.float32)
    conv_codes = pd.factorize(X["Conveyor_ID"])[0]
    data = []
    for tr, te in folds:
        itr = X["Conveyor_ID"].isin(tr).to_numpy()
        ite = X["Conveyor_ID"].isin(te).to_numpy()
        per_k = []
        for k in ks:
            t = Y[f"t{k}"].to_numpy(dtype=float)
            ev = Y[f"e{k}"].to_numpy() == 1
            lo = np.maximum(t, 0.5)
            hi = np.where(ev, lo, np.inf)
            dtr = xgb.QuantileDMatrix(Xf[itr], weight=w[itr], label_lower_bound=lo[itr], label_upper_bound=hi[itr])
            sel = ite & ev
            codes, uniq = pd.factorize(conv_codes[sel])
            per_k.append(dict(dtr=dtr, dte=xgb.DMatrix(Xf[sel]), true=t[sel], codes=codes, n_conv=len(uniq)))
        data.append(per_k)
        print("  prepared fold", flush=True)
    return data


def aft_objective(data, threads):
    def objective(trial):
        lr = trial.suggest_float("learning_rate", 0.01, 0.2, log=True)
        p = {**XGB_AFT, "learning_rate": lr, "nthread": threads, "seed": 0,
             "device": os.environ.get("CONVEYOR_XGB_DEVICE", "cpu"),
             "max_depth": trial.suggest_int("max_depth", 3, 10),
             "min_child_weight": trial.suggest_float("min_child_weight", 1, 300, log=True),
             "subsample": trial.suggest_float("subsample", 0.5, 1.0),
             "colsample_bytree": trial.suggest_float("colsample_bytree", 0.4, 1.0),
             "lambda": trial.suggest_float("lambda", 1e-3, 30, log=True),
             "aft_loss_distribution": trial.suggest_categorical("aft_loss_distribution", ["normal", "logistic", "extreme"]),
             "aft_loss_distribution_scale": trial.suggest_float("aft_loss_distribution_scale", 0.3, 2.0, log=True)}
        max_rounds = int(min(2000, 25 / lr))
        grid = np.unique(np.linspace(max_rounds / 20, max_rounds, 20).astype(int))
        curves = []
        for per_k in data:
            fold_curve = np.zeros(len(grid))
            for d in per_k:
                bst = xgb.train(p, d["dtr"], max_rounds)
                for i, r in enumerate(grid):
                    pred = bst.predict(d["dte"], iteration_range=(0, int(r)))
                    err = np.abs(np.log1p(pred) - np.log1p(d["true"]))
                    fold_curve[i] += conv_mean(err, d["codes"], d["n_conv"])
            curves.append(fold_curve)
        curve = np.mean(curves, axis=0)
        best = int(np.argmin(curve))
        trial.set_user_attr("rounds", int(grid[best]))
        return float(curve[best])
    return objective


def aft_default():
    return {"learning_rate": 0.05, "max_depth": 6, "min_child_weight": 20, "subsample": 0.8, "colsample_bytree": 0.8,
            "lambda": 1.0, "aft_loss_distribution": "normal", "aft_loss_distribution_scale": 1.0}


# --- comp -------------------------------------------------------------------------------------
def prep_comp(X, Y, folds, w, k=1):
    Xf = X[feature_cols(X)].to_numpy(dtype=np.float32)
    conv_codes = pd.factorize(X["Conveyor_ID"])[0]
    ev = Y[f"e{k}"].to_numpy() == 1
    c = Y[f"c{k}"].to_numpy()
    data = []
    for tr, te in folds:
        itr = X["Conveyor_ID"].isin(tr).to_numpy() & ev
        ite = X["Conveyor_ID"].isin(te).to_numpy() & ev
        dtr = lgb.Dataset(Xf[itr], c[itr], weight=w[itr], params={"feature_pre_filter": False, "verbose": -1}).construct()
        dte = lgb.Dataset(Xf[ite], c[ite], weight=w[ite], reference=dtr).construct()
        codes, uniq = pd.factorize(conv_codes[ite])
        data.append(dict(dtr=dtr, dte=dte, Xte=Xf[ite], true=c[ite], codes=codes, n_conv=len(uniq)))
    print("  prepared folds", flush=True)
    return data


def comp_objective(data, threads):
    def objective(trial):
        lr = trial.suggest_float("learning_rate", 0.01, 0.15, log=True)
        p = {**LGB_COMP, "learning_rate": lr, "metric": "multi_logloss", "num_threads": threads, "seed": 0,
             "num_leaves": trial.suggest_int("num_leaves", 7, 127, log=True),
             "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 50, 5000, log=True),
             "feature_fraction": trial.suggest_float("feature_fraction", 0.3, 1.0),
             "bagging_fraction": trial.suggest_float("bagging_fraction", 0.5, 1.0),
             "lambda_l2": trial.suggest_float("lambda_l2", 1e-3, 100, log=True)}
        max_rounds = int(min(2000, 20 / lr))
        curves, boosters = [], []
        for f in data:
            rec = {}
            boosters.append(lgb.train(p, f["dtr"], max_rounds, valid_sets=[f["dte"]], valid_names=["te"],
                                      callbacks=[lgb.record_evaluation(rec)]))
            curves.append(rec["te"]["multi_logloss"])
        curve = np.mean(curves, axis=0)
        best = int(np.argmin(curve))
        acc = np.mean([conv_mean((b.predict(f["Xte"], num_iteration=best + 1).argmax(1) == f["true"]).astype(float),
                                 f["codes"], f["n_conv"]) for b, f in zip(boosters, data)])
        base = np.mean([conv_mean((f["true"] == 0).astype(float), f["codes"], f["n_conv"]) for f in data])
        trial.set_user_attr("rounds", best + 1)
        trial.set_user_attr("c1_acc", float(acc))
        trial.set_user_attr("c1_acc_always_bearing", float(base))
        return float(curve[best])
    return objective


def comp_default():
    return {k: LGB_COMP[k] for k in ("learning_rate", "num_leaves", "min_data_in_leaf", "feature_fraction",
                                     "bagging_fraction", "lambda_l2")}


STUDIES = {"count": (prep_count, count_objective, count_default, 600),
           "aft": (prep_aft, aft_objective, aft_default, 400),
           "comp": (prep_comp, comp_objective, comp_default, 300)}


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("sub", choices=list(STUDIES))
    ap.add_argument("--timeout", type=int, default=2400)
    ap.add_argument("--threads", type=int, default=8)
    ap.add_argument("--folds", type=int, default=3)
    ap.add_argument("--sampler-seed", type=int, default=0, help="use a different seed for each extra parallel worker")
    a = ap.parse_args()
    prep, make_obj, default, default_rounds = STUDIES[a.sub]
    t0 = time.time()
    X, Y, folds, w = load(a.folds)
    data = prep(X, Y, folds, w)
    del X, Y
    print(f"[{a.sub}] data ready in {time.time()-t0:.0f}s", flush=True)

    OUT.mkdir(parents=True, exist_ok=True)
    storage = f"sqlite:///{(CACHE / 'optuna.db').as_posix()}"
    study = optuna.create_study(study_name=f"{a.sub}_f{a.folds}", storage=storage, direction="minimize",
                                sampler=optuna.samplers.TPESampler(seed=a.sampler_seed, n_startup_trials=8), load_if_exists=True)
    if len(study.trials) == 0:
        study.enqueue_trial(default())

    def save(study, trial):
        if trial.state != optuna.trial.TrialState.COMPLETE:
            return
        print(f"[{a.sub}] trial {trial.number}: {trial.value:.5f} rounds={trial.user_attrs.get('rounds')} "
              f"{({k: v for k, v in trial.user_attrs.items() if k != 'rounds'})} ({time.time()-t0:.0f}s)", flush=True)
        b = study.best_trial
        (OUT / f"{a.sub}.json").write_text(json.dumps(
            {"params": b.params, "rounds": b.user_attrs["rounds"], "cv_value": b.value, "trial": b.number,
             "default_value": study.trials[0].value, "n_trials": len(study.trials), "user_attrs": b.user_attrs,
             "harness": f"grouped CV, {a.folds} folds, seed 0, non-sealed conveyors"}, indent=1))

    optuna.logging.set_verbosity(optuna.logging.WARNING)
    study.optimize(make_obj(data, a.threads), timeout=a.timeout, callbacks=[save], gc_after_trial=True)
    b = study.best_trial
    print(f"[{a.sub}] BEST {b.value:.5f} (default {study.trials[0].value:.5f}) rounds={b.user_attrs['rounds']} {b.params}")


if __name__ == "__main__":
    main()
